## Final Load Prep

Calculating KPIs and exporting summary tables for Tableau.

In [ ]:
import pandas as pd
import os

df = pd.read_csv('../data/processed/cleaned_retail_data.csv')
active = df[df['IsCancelled']==False].copy()
out = '../data/processed/'
print('loaded', df.shape)

In [ ]:
# headline numbers
total_rev = active['TotalRevenue'].sum()
total_orders = active['InvoiceNo'].nunique()
total_customers = active[active['CustomerID']!='Guest']['CustomerID'].nunique()
aov = total_rev / total_orders
cancel_rate = df['IsCancelled'].mean() * 100

print(f'Total Revenue     : £{total_rev:,.0f}')
print(f'Total Orders      : {total_orders:,}')
print(f'Unique Customers  : {total_customers:,}')
print(f'Avg Order Value   : £{aov:.2f}')
print(f'Cancellation Rate : {cancel_rate:.2f}%')

In [ ]:
# monthly kpi table
monthly = active.groupby(['Year','MonthNumber','MonthName','InvoiceYearMonth','Quarter']).agg(
    Revenue=('TotalRevenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique'),
    ItemsSold=('Quantity','sum')
).reset_index().sort_values(['Year','MonthNumber'])
monthly['AvgOrderValue'] = (monthly['Revenue']/monthly['Orders']).round(2)
monthly.to_csv(out+'tableau_monthly_kpi.csv', index=False)
print('monthly done')
monthly.head()

In [ ]:
# country table
country = active.groupby(['Country','IsUK']).agg(
    Revenue=('TotalRevenue','sum'),
    Orders=('InvoiceNo','nunique'),
    Customers=('CustomerID','nunique')
).reset_index().sort_values('Revenue', ascending=False)
country['AvgOrderValue'] = (country['Revenue']/country['Orders']).round(2)
country.to_csv(out+'tableau_country_kpi.csv', index=False)
print('country done')
country.head(10)

In [ ]:
# product table
products = active.groupby(['StockCode','Description']).agg(
    Revenue=('TotalRevenue','sum'),
    QuantitySold=('Quantity','sum'),
    Orders=('InvoiceNo','nunique')
).reset_index().sort_values('Revenue', ascending=False)
products.to_csv(out+'tableau_product_kpi.csv', index=False)
print('products done')

In [ ]:
# customer segments
segments = (active[active['CustomerID']!='Guest']
            .drop_duplicates('CustomerID')
            .groupby('CustomerSegment')
            .agg(Customers=('CustomerID','nunique'))
            .reset_index())
segments['Pct'] = (segments['Customers']/segments['Customers'].sum()*100).round(1)
segments.to_csv(out+'tableau_segment_kpi.csv', index=False)
print(segments)

In [ ]:
# time heatmap data
time_data = active.groupby(['DayOfWeek','TimeOfDay','Hour']).agg(
    Revenue=('TotalRevenue','sum'),
    Orders=('InvoiceNo','nunique')
).reset_index()
time_data.to_csv(out+'tableau_time_kpi.csv', index=False)
print('time data done')

In [ ]:
# basket size summary
basket_sum = (active.drop_duplicates('InvoiceNo')
              .groupby('BasketSize')
              .agg(Orders=('InvoiceNo','count'), Revenue=('TotalRevenue','sum'))
              .reset_index())
basket_sum['AvgOrderValue'] = (basket_sum['Revenue']/basket_sum['Orders']).round(2)
basket_sum.to_csv(out+'tableau_basket_kpi.csv', index=False)
print('basket done')
print(basket_sum)

In [ ]:
print('all files saved:')
for f in sorted(os.listdir(out)):
    print(' ', f, '-', round(os.path.getsize(out+f)/1024, 1), 'KB')